In [ ]:
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"
import sympy as sym

from rotation.symbolic.general import ANGLE, AXIAL_COUPLING, MASS, create_3d_rotation
from rotation.symbolic.axial import kinetic_sines, create_kinetic, create_axial_mix_equation, bare_axial_meson_couplings, rotated_axial_meson_couplings, bare_axial_masses, bare_axial_kaon_couplings, rotated_axial_kaon_couplings

## Mixing Equation

In [ ]:
particles = tuple(["pi0", "eta", "etaP"])

In [ ]:
# Define some labels to increase clarity of output
rotate_2d_symbol = sym.MatrixSymbol(r"R_{2D}", 2, 2)
rotation_symbols = []
for name in [r"R_\alpha", r"R_\beta", r"R_\gamma"]:
    rotation_symbols.append(sym.MatrixSymbol(name, 3, 3))
rotate_symbol = sym.MatrixSymbol("R", 3, 3)
matrix_symbols = []
for name in [r"R^\text{T}", r"M_b", "M"]:
    matrix_symbols.append(sym.MatrixSymbol(name, 3, 3))

In [ ]:
rotate, rotations, rotate_2d = create_3d_rotation()
kin = create_kinetic()
axial_equation, axial_matrices = create_axial_mix_equation(rotate, kin)

In [ ]:
# Rotation
print("The 2D rotation matrix is:")
sym.Eq(rotate_2d_symbol, rotate_2d, evaluate=False)
print("Which is used to create 3 different 3D rotations:")
for symbol, rot in zip(rotation_symbols, rotations):
    sym.Eq(symbol, rot, evaluate=False)
print("And their multiplication creates the general 3D rotation matrix:")
sym.Eq(
    rotate_symbol,
    sym.MatMul(*rotation_symbols, evaluate=False),
    evaluate=False,
)
sym.Eq(
    rotate_symbol, sym.MatMul(*rotations, evaluate=False), evaluate=False
)

In [ ]:
print("The axial vector equation uses the kinetic rotation matrix in addition:")
kinetic_symbol = sym.MatrixSymbol(r"R_\text{kin}", 3, 3)
kinetic_T_symbol = sym.MatrixSymbol(r"R_\text{kin}^T", 3, 3)
for s, eq in zip([sym.sin(ANGLE[k]) for k in ["u", "v", "w"]], kinetic_sines()):
    sym.Eq(s, eq, evaluate=False)
for s, mat in zip([kinetic_symbol, rotate_symbol, kinetic_T_symbol] + matrix_symbols, [kin, rotate]+list(axial_matrices)):
    sym.Eq(s, mat, evaluate=False)
print("Which are all used in the mixing equation:")
sym.Eq(
    matrix_symbols[2],
    sym.MatMul(rotate_symbol, kinetic_T_symbol, matrix_symbols[1], kinetic_symbol, matrix_symbols[0]),
    evaluate=False,
)
axial_equation
sym.Eq(
    axial_equation.lhs,
    1*axial_equation.rhs,
    evaluate=False,
)

In [ ]:
print("The following are the off-diagonal bare mass terms:")
for s, g in zip([MASS[f"{k}-AP"] for k in particles], bare_axial_masses()):
    sym.Eq(s, g, evaluate=False)

## Couplings

In [ ]:
print("The following are the un-rotated dark matter to meson couplings:")
for s, g in zip([AXIAL_COUPLING[f"{k}Bare-chi"] for k in particles], bare_axial_meson_couplings()):
    sym.Eq(s, g, evaluate=False)

In [ ]:
print("The following are the rotated dark matter to meson couplings:")
for s, g in zip([AXIAL_COUPLING[f"{k}-chi"] for k in particles], rotated_axial_meson_couplings()):
    sym.Eq(s, g, evaluate=False)

In [ ]:
print("Substituting the bare mass off-diagonals into the rotated dark matter to meson couplings:")
for s, g in zip([AXIAL_COUPLING[f"{k}-chi"] for k in particles], rotated_axial_meson_couplings()):
    subbed = g.subs({MASS[f"{k}-AP"]:v for k,v in zip(particles, bare_axial_masses())})
    sym.Eq(s, subbed, evaluate=False)

In [ ]:
print("The following are the un-rotated bare pion-pion-kaon, eta-pion-kaon, and etaprime-pion-kaon couplings:")
for s, g in zip([AXIAL_COUPLING[f"{k}Bare-pi-K"] for k in particles], bare_axial_kaon_couplings()):
    sym.Eq(s, g, evaluate=False)

In [ ]:
print("The following are the rotated kaon couplings:")
for s, g in zip([AXIAL_COUPLING[f"{k}-pi-K"] for k in particles], rotated_axial_kaon_couplings()):
    sym.Eq(s, g, evaluate=False)

In [ ]:
print("Substituting the un-rotated couplings and bare mass off-diagonals into the rotated kaon couplings:")
for s, g in zip([AXIAL_COUPLING[f"{k}-pi-K"] for k in particles], rotated_axial_kaon_couplings()):
    subbed = g.subs({AXIAL_COUPLING[f"{k}Bare-pi-K"]:v for k,v in zip(particles, bare_axial_kaon_couplings())})
    subbed = subbed.subs({MASS[f"{k}-AP"]:v for k,v in zip(particles, bare_axial_masses())})
    sym.Eq(s, subbed, evaluate=False)